## Construct test cases
First set up the waveform and response modules for injection and recovery. 

In [1]:
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt
from lisaconstants import ASTRONOMICAL_YEAR
from lisaorbits import OEMOrbits
from mojito import MojitoL1File
from ruamel.yaml import YAML
from scipy.signal.windows import tukey

from src.noise import build_inv_covariance
from src.utils import inband_freqs, inner_prod_tdi, mismatch_tdi
from src.waveform import (
    ResponseConfig,
    WaveformConfig,
    build_response,
    param_names_for,
)

/data/leuven/367/vsc36785/miniconda3/envs/emri_env_ddpc/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#File path constants
ORBIT_FILE = "/data/leuven/367/vsc36785/LISA/Mojito_analysis/esa-trailing-orbits-mojito_validation_test_2.h5"
MOJITO_L1  = (
    "/scratch/leuven/367/vsc36785/MojitoLight/SIM_data/brickmarket/"
    "mojito_light_v1_0_0/data/EMRI/L1/"
    "EMRI_731d_2.5s_L1_source0_0_20251203T225446987631Z.h5"
)
NOISE_FILE = (
    "/scratch/leuven/367/vsc36785/MojitoLight/SIM_data/brickmarket/"
    "mojito_light_v1_0_0/data/NOISE/L1/"
    "NOISE_731d_2.5s_L1_source0_0_20251206T220508924302Z.h5"
)


In [3]:

# Mojito L1 timing
with MojitoL1File(MOJITO_L1) as l1:
    ts           = l1.tdis.time_sampling
    t0_l1        = float(ts.t0)
    mojito_dt    = float(ts.dt)
    central_freq = float(l1.laser_frequency)

print(f"Mojito L1:  t0={t0_l1:.3f} s   dt={mojito_dt:.3f} s   f_laser={central_freq:.6e} Hz")

# Injection waveform config
inj_wcfg = WaveformConfig(
    model="1PAT1R",
    dt=5.0,
    T=2.0,
    evolve_chi1=False,
    include_1PA_amps=False,
    inspiral_kwargs={"DENSE_STEPPING": 0, "max_init_len": 1000},
    summation_kwargs={"pad_output": True},
    amplitude_kwargs={},
)
print(f"Injection : {inj_wcfg.model}  evolve_chi1={inj_wcfg.evolve_chi1}  1PA_amps={inj_wcfg.include_1PA_amps}")

Mojito L1:  t0=97729939.828 s   dt=2.500 s   f_laser=2.816000e+14 Hz
Injection : 1PAT1R  evolve_chi1=False  1PA_amps=False


In [23]:
#Recovery waveform config
rec_wcfg = WaveformConfig(
    model="1PAT1R",
    dt=5.0,
    T=2.0,
    evolve_chi1=True,
    include_1PA_amps=True,
    inspiral_kwargs={"DENSE_STEPPING": 0, "max_init_len": 1000},
    summation_kwargs={"pad_output": True},
    amplitude_kwargs={},
)
print(f"Recovery  : {rec_wcfg.model}  evolve_chi1={rec_wcfg.evolve_chi1}  1PA_amps={rec_wcfg.include_1PA_amps}")
if inj_wcfg.model != rec_wcfg.model:
    print(" Models differ — systematic bias is expected.")


Recovery  : 1PAT1R  evolve_chi1=True  1PA_amps=True


In [24]:

# Shared response config (same for inj and rec to avoid response artifacts)
resp_cfg = ResponseConfig(
    orbit_file=ORBIT_FILE,
    tdi_gen="2nd generation",
    tdi_chan="XYZ",
    order=40,
    offset=550.0,
    n_samples_delay=1000,
    t_buffer=10000.0,
    flip_hx=True,
    is_ecliptic_latitude=False,
)
print(f"Response  : {resp_cfg.tdi_chan}  {resp_cfg.tdi_gen}  order={resp_cfg.order}")

Response  : XYZ  2nd generation  order=40


In [25]:
# Derived timing (mirrors PE_response.py)
DT         = inj_wcfg.dt
oem_orbits = OEMOrbits.from_included("esa-trailing")
t0_orbits  = float(oem_orbits.t_start) + 10.0
T_response = (
    inj_wcfg.T
    + (2 * resp_cfg.offset + 2 * resp_cfg.n_samples_delay * DT) / ASTRONOMICAL_YEAR
)
t0_l0  = t0_l1 - resp_cfg.n_samples_delay * mojito_dt
t_init = t0_l0 - resp_cfg.offset

print(f"t_init     = {t_init:.3f} s")
print(f"T_response = {T_response:.6f} yr")

# Build injection and recovery response callables
print("\nBuilding injection response …")
inj_response = build_response(inj_wcfg, resp_cfg, t_init, t0_orbits, T_response, use_gpu=True)

print("Building recovery response …")
rec_response = build_response(rec_wcfg, resp_cfg, t_init, t0_orbits, T_response, use_gpu=True)

print("Done.")

OEM preferred interpolation method ignored, using spline interpolation (see InterpolatedOrbits for details)


t_init     = 97726889.828 s
T_response = 2.000352 yr

Building injection response …
Building recovery response …
Done.


## Mismatch code
Compute mismatches between two datasets. Gives an indication of the likes of the mismatch that we can expect. 

In [26]:
# inner_prod_tdi and mismatch_tdi are imported from src.utils and work on
# (n_chan, n_f) complex arrays with (n_f, n_chan, n_chan) inv_cov.

def snr_tdi(h_fft: cp.ndarray, inv_cov: cp.ndarray) -> float:
    return float(cp.sqrt(inner_prod_tdi(h_fft, h_fft, inv_cov)))

def _to_vector(params: dict, model: str) -> list:
    """Convert a params dict to the ordered vector expected by the waveform model."""
    z = float(params.get("z", 0.0))
    vec = []
    for n in param_names_for(model):
        val = float(params[n])
        if n in ("M", "mu"):
            val *= (1.0 + z)
        vec.append(val)
    return vec

In [27]:
# Nominal N_t from timing config — used to set up the frequency grid.
# If the actual waveform length differs slightly (due to ResponseWrapper padding),
# inv_cov is rebuilt automatically in the mismatch cell below.
N_t_nominal = int(round(T_response * ASTRONOMICAL_YEAR / DT))
freqs_inband_nom, mask_nom = inband_freqs(N_t_nominal, DT, filter_freq=True)

print(f"N_t_nominal = {N_t_nominal}   n_inband = {int(mask_nom.sum())}")
print("Building inverse covariance …")

inv_cov, psd_diag = build_inv_covariance(
    NOISE_FILE, central_freq,
    cp.asnumpy(freqs_inband_nom), DT, N_t_nominal,
    channels=resp_cfg.tdi_chan,
)
print(f"inv_cov shape: {inv_cov.shape}")

N_t_nominal = 12625480   n_inband = 6312109
Building inverse covariance …
inv_cov shape: (6312109, 3, 3)


In [28]:
# Intrinsic parameters: tune these
M   = 1000000.0   # primary mass [M_sun], source frame
mu  = 6.0     # secondary mass [M_sun], source frame
a   = 0.00002    # primary spin [-0.999 … 0.999]
p0  = 10.0    # initial semi-latus rectum [M]
e0  = 0.0     # initial eccentricity
chi2 = 0.9   # secondary spin (only used by 1PAT1R recovery)
# function to compute luminosity distance from redshift:
z   = 0.5   # redshift (masses are redshifted by (1+z) internally)

d_L = 0.1    # luminosity distance [Gpc]


# Extrinsic / angular parameters: drawn randomly
rng = np.random.default_rng(seed=42)

theta_S    = rng.uniform(0.0, np.pi)       # source ecliptic co-latitude
phi_S      = rng.uniform(0.0, 2 * np.pi)  # source ecliptic longitude
theta_K    = rng.uniform(0.0, np.pi)       # spin co-latitude
phi_K      = rng.uniform(0.0, 2 * np.pi)  # spin longitude
Phi_phi0   = rng.uniform(0.0, 2 * np.pi)  # initial azimuthal phase
Phi_theta0 = rng.uniform(0.0, 2 * np.pi)  # initial polar phase
Phi_r0     = rng.uniform(0.0, 2 * np.pi)  # initial radial phase

inj_params = dict(
    M=M, 
    mu=mu, 
    a=a, 
    p0=p0, 
    e0=e0, 
    chi2=chi2,
    x_I0=1.0, 
    d_L=d_L,
    theta_S=theta_S, 
    phi_S=phi_S,
    theta_K=theta_K, 
    phi_K=phi_K,
    Phi_phi0=Phi_phi0, 
    Phi_theta0=Phi_theta0, 
    Phi_r0=Phi_r0,
    z=z,
)

print("Injection params:")
for k, v in inj_params.items():
    print(f"  {k:12s} = {v:.6g}")

Injection params:
  M            = 1e+06
  mu           = 6
  a            = 2e-05
  p0           = 10
  e0           = 0
  chi2         = 0.9
  x_I0         = 1
  d_L          = 0.1
  theta_S      = 2.43145
  phi_S        = 2.75755
  theta_K      = 2.69736
  phi_K        = 4.38169
  Phi_phi0     = 0.591734
  Phi_theta0   = 6.13002
  Phi_r0       = 4.78238
  z            = 0.5


In [29]:
# Recovery params represent the same physical source in the recovery model's
# parameter space. Usually identical to inj_params; edit here to probe systematics.
rec_params = dict(inj_params)

In [30]:
windowing  = True

inj_vec = _to_vector(inj_params, inj_wcfg.model)
rec_vec = _to_vector(rec_params, rec_wcfg.model)

# Generate injection TDI
print("Generating injection TDI …")
xyz_inj = inj_response(*inj_vec)
N_t     = xyz_inj.shape[1]
window  = cp.asarray(tukey(N_t, alpha=0.01)) if windowing else cp.ones(N_t)
freqs_inband, mask = inband_freqs(N_t, DT, filter_freq=True)
xyz_inj_fft = cp.fft.rfft(xyz_inj * window, axis=1)[:, mask]
print(f"  N_t = {N_t}   n_inband = {int(mask.sum())}")

# Rebuild inv_cov on the exact grid if N_t differed from the nominal estimate
if N_t != N_t_nominal:
    print(f"  N_t differs from nominal ({N_t_nominal}) — rebuilding inv_cov …")
    inv_cov, psd_diag = build_inv_covariance(
        NOISE_FILE, central_freq,
        cp.asnumpy(freqs_inband), DT, N_t,
        channels=resp_cfg.tdi_chan,
    )


Generating injection TDI …
  N_t = 12625479   n_inband = 6312108
  N_t differs from nominal (12625480) — rebuilding inv_cov …


In [31]:

# Generate recovery TDI at injection params
print("Generating recovery TDI at injection params …")
xyz_rec     = rec_response(*rec_vec)
xyz_rec_fft = cp.fft.rfft(xyz_rec * window, axis=1)[:, mask]

# Diagnostics
snr_inj = snr_tdi(xyz_inj_fft, inv_cov)
mm       = mismatch_tdi(xyz_inj_fft, xyz_rec_fft, inv_cov)

xyz_res_fft = xyz_inj_fft - xyz_rec_fft
snr_res     = snr_tdi(xyz_res_fft, inv_cov)

print(f"\nInjection SNR           : {snr_inj:.2f}")
print(f"Mismatch (inj vs rec)   : {mm:.3e}")
print(f"SNR of residual         : {snr_res:.2f}")

if snr_inj < 20:
    print("WARNING: SNR below 20 — source may not be recoverable.")

Generating recovery TDI at injection params …

Injection SNR           : 255.52
Mismatch (inj vs rec)   : 1.826e-11
SNR of residual         : 0.00


## Write to a config file
If satisfied with the parameters, write to a config file automatically. 

In [ ]:
import os

CONFIG_NAME  = "config_test_2"       
CONFIG_DIR   = "config"
FIXED_PARAMS = ["x_I0", "e0"]             

os.makedirs(CONFIG_DIR, exist_ok=True)

# Construct yaml structure, can be written to file 
cfg_out = {
    "Data": {
        "orbit_file":     ORBIT_FILE,
        "mojito_l1_file": MOJITO_L1,
        "noise_file":     NOISE_FILE,
    },
    "Injection": {
        "EMRI": {k: float(v) for k, v in inj_params.items()},
        "Waveform": {
            "model":                    inj_wcfg.model,
            "evolve_chi1":              inj_wcfg.evolve_chi1,
            "include_1PA_amps":         inj_wcfg.include_1PA_amps,
            "dt":                       inj_wcfg.dt,
            "T":                        inj_wcfg.T,
            "mode_selection_threshold": inj_wcfg.mode_selection_threshold,
            "inspiral_kwargs":          dict(inj_wcfg.inspiral_kwargs),
            "summation_kwargs":         dict(inj_wcfg.summation_kwargs),
            "amplitude_kwargs":         dict(inj_wcfg.amplitude_kwargs),
        },
    },
    "Response": {
        "tdi_gen":             resp_cfg.tdi_gen,
        "tdi_chan":            resp_cfg.tdi_chan,
        "order":               resp_cfg.order,
        "offset":              resp_cfg.offset,
        "n_samples_delay":     resp_cfg.n_samples_delay,
        "t_buffer":            resp_cfg.t_buffer,
        "flip_hx":             resp_cfg.flip_hx,
        "is_ecliptic_latitude": resp_cfg.is_ecliptic_latitude,
    },
    "Recovery": {
        "Waveform": {
            "model":                    rec_wcfg.model,
            "evolve_chi1":              rec_wcfg.evolve_chi1,
            "include_1PA_amps":         rec_wcfg.include_1PA_amps,
            "dt":                       rec_wcfg.dt,
            "T":                        rec_wcfg.T,
            "mode_selection_threshold": rec_wcfg.mode_selection_threshold,
            "inspiral_kwargs":          dict(rec_wcfg.inspiral_kwargs),
            "summation_kwargs":         dict(rec_wcfg.summation_kwargs),
            "amplitude_kwargs":         dict(rec_wcfg.amplitude_kwargs),
        },
    },
    "Sampler": {
        "name":              CONFIG_NAME,
        "windowing":         True,
        "filter_freq":       True,
        "num_samples":       10000,
        "burn_in":           0,
        "use_gpu":           True,
        "n_temps":           3,
        "n_walkers":         30,
        "d":                 5,
        "continue_run":      False,
        "sampling_data_path": "../sampling_data",
        "plots_path":        "../Plots",
        "fixed_params":      FIXED_PARAMS,
    },
}


In [ ]:

yaml = YAML()
yaml.default_flow_style = False
yaml.indent(mapping=2, sequence=4, offset=2)

out_path = os.path.join(CONFIG_DIR, f"{CONFIG_NAME}.yaml")
with open(out_path, "w") as f:
    yaml.dump(cfg_out, f)

print(f"Config written to  : {out_path}")
print(f"SNR                : {snr_inj:.2f}")
print(f"Mismatch (inj|rec) : {mm:.3e}")
print(f"SNR residual       : {snr_res:.2f}")